# Python 可变对象、引用与拷贝

## 学习目标

通过可操作的对象关系图理解赋值、浅拷贝、深拷贝、函数参数和可变默认参数。
重点不是记住几个 API，而是建立一个心智模型：

> Python 变量保存的是对象引用；复制操作决定新旧容器在哪一层继续共享对象。

## 准备

本教程使用 `ipywidgets` 改变复制方式，使用 Matplotlib 画出变量、外层列表和
内层列表之间的引用关系。图中的 A、B、C 只是为了区分对象，不代表变量名。

In [ ]:
%matplotlib inline

import ipywidgets as widgets
import matplotlib.pyplot as plt
from IPython.display import clear_output, display

plt.rcParams["axes.unicode_minus"] = False

In [ ]:
import copy


def inspect_value(name, value):
    print(
        f"{name:<10} value={value!r:<24} "
        f"type={type(value).__name__:<8} id={id(value)}"
    )

## 先建立心智模型

### 1. 赋值不是复制

`alias = original` 只增加一个变量名。两个名字仍指向同一个外层列表，因此修改任何一方
都会被另一方观察到。

In [ ]:
original = [[1, 2], [3, 4]]
alias = original

inspect_value("original", original)
inspect_value("alias", alias)
print("同一个对象:", original is alias)

### 2. 直接操作引用关系图

在下拉框中切换三种方式：

- **直接赋值**：外层和内层都共享；
- **浅拷贝**：外层分开，内层继续共享；
- **深拷贝**：本例中的外层和内层都分开。

先看箭头，再看图下方的两个 `is` 结论。真正决定修改是否互相影响的是“被修改的那一层
是否仍然指向同一个对象”。

In [ ]:
def unique_objects(values):
    unique = []
    for value in values:
        if not any(value is existing for existing in unique):
            unique.append(value)
    return unique


def vertical_positions(count):
    return [(count - index) / (count + 1) for index in range(count)]


def draw_box(axis, x, y, text, color):
    axis.text(
        x,
        y,
        text,
        ha="center",
        va="center",
        fontsize=10,
        bbox={
            "boxstyle": "round,pad=0.45",
            "facecolor": color,
            "edgecolor": "#334155",
        },
    )


def draw_arrow(axis, start, end):
    axis.annotate(
        "",
        xy=end,
        xytext=start,
        arrowprops={"arrowstyle": "->", "color": "#475569", "lw": 1.8},
    )


def build_copy_example(copy_mode):
    source = [[1, 2], [3, 4]]
    if copy_mode == "直接赋值":
        copied = source
    elif copy_mode == "浅拷贝":
        copied = source.copy()
    else:
        copied = copy.deepcopy(source)
    return source, copied


def draw_copy_graph(copy_mode):
    source, copied = build_copy_example(copy_mode)
    mode_labels = {
        "直接赋值": "assignment",
        "浅拷贝": "shallow copy",
        "深拷贝": "deep copy",
    }
    outer_objects = unique_objects([source, copied])
    inner_objects = unique_objects(
        [item for outer in outer_objects for item in outer]
    )

    outer_y = {
        id(value): y
        for value, y in zip(
            outer_objects,
            vertical_positions(len(outer_objects)),
            strict=True,
        )
    }
    inner_y = {
        id(value): y
        for value, y in zip(
            inner_objects,
            vertical_positions(len(inner_objects)),
            strict=True,
        )
    }

    figure, axis = plt.subplots(figsize=(10, 5))
    axis.set_xlim(0, 1)
    axis.set_ylim(0, 1)
    axis.axis("off")
    axis.set_title(
        f"Object references after {mode_labels[copy_mode]}",
        fontsize=15,
        pad=16,
    )

    variable_positions = {"original": 0.7, "copied": 0.3}
    variable_targets = {"original": source, "copied": copied}
    for name, y in variable_positions.items():
        draw_box(axis, 0.1, y, name, "#e2e8f0")
        target_y = outer_y[id(variable_targets[name])]
        draw_arrow(axis, (0.18, y), (0.32, target_y))

    for index, value in enumerate(outer_objects):
        y = outer_y[id(value)]
        label = chr(ord("A") + index)
        draw_box(axis, 0.42, y, f"outer list {label}\n{value!r}", "#bfdbfe")
        for inner in value:
            draw_arrow(axis, (0.52, y), (0.68, inner_y[id(inner)]))

    for index, value in enumerate(inner_objects):
        y = inner_y[id(value)]
        label = chr(ord("A") + index)
        draw_box(axis, 0.8, y, f"nested list {label}\n{value!r}", "#fed7aa")

    axis.text(0.1, 0.96, "names", ha="center", weight="bold")
    axis.text(0.42, 0.96, "outer objects", ha="center", weight="bold")
    axis.text(0.8, 0.96, "nested objects", ha="center", weight="bold")
    axis.text(
        0.5,
        0.02,
        (
            f"same outer object: {source is copied}    "
            f"same first nested object: {source[0] is copied[0]}"
        ),
        ha="center",
        fontsize=11,
        color="#0f172a",
    )
    plt.show()
    plt.close(figure)


copy_mode = widgets.Dropdown(
    options=["直接赋值", "浅拷贝", "深拷贝"],
    value="浅拷贝",
    description="复制方式：",
    style={"description_width": "initial"},
)
copy_output = widgets.Output()


def render_copy_mode(_change=None):
    with copy_output:
        clear_output(wait=True)
        draw_copy_graph(copy_mode.value)


copy_mode.observe(render_copy_mode, names="value")
display(widgets.VBox([copy_mode, copy_output]))
render_copy_mode()

### 3. 用修改实验验证图中的箭头

下面先修改内层列表，再比较三种方式。不要笼统地说“浅拷贝会联动”，应准确地说：
**浅拷贝得到的新外层列表仍然引用原来的内层列表。**

In [ ]:
for mode in ["直接赋值", "浅拷贝", "深拷贝"]:
    source, copied = build_copy_example(mode)
    source[0].append(99)
    print(f"{mode:<4} source={source!r} copied={copied!r}")

## 函数参数仍然遵循同一模型

重新绑定局部变量不会改变调用者的名字，但修改可变对象本身会被调用者观察到。
这不是另一套特殊规则：形参只是函数调用期间新创建的局部变量名。

In [ ]:
def rebind(items):
    items = ["函数中的新列表"]
    return items


def mutate(items):
    items.append("函数添加")


values = ["原值"]
rebound = rebind(values)
print("重新绑定后:", values, rebound)

mutate(values)
print("原地修改后:", values)

## 可变默认参数为什么会复用

默认参数在函数定义时创建一次，而不是每次调用时创建。

In [ ]:
def append_bad(value, items=[]):  # noqa: B006 - 故意演示反例
    items.append(value)
    return items


def append_good(value, items=None):
    if items is None:
        items = []
    items.append(value)
    return items


print("错误:", append_bad("A"), append_bad("B"))
print("正确:", append_good("A"), append_good("B"))

## 常见误解

- “赋值会复制对象”：赋值只让另一个名字指向对象。
- “浅拷贝一定不安全”：风险取决于是否修改了继续共享的嵌套对象。
- “深拷贝永远正确”：深拷贝成本更高，而且外部资源、单例和自定义复制语义需要单独考虑。
- “Python 是传值还是传引用”：更准确的说法是把对象引用绑定给新的局部形参。

## 面试时可以这样解释

> Python 变量是名字到对象的绑定。浅拷贝创建新的外层容器，但元素引用被原样复制；
> 深拷贝会递归复制可复制的子对象。因此判断修改是否互相影响时，
> 要看被修改层级是否共享。

## 继续探索

1. 把内层列表换成不可变元组，观察浅拷贝是否仍有风险。
2. 给列表中加入自定义类实例，再比较三种复制方式。
3. 在 VS Code 变量面板中观察外层和内层对象的 `id()`。